In [0]:
# DBTITLE 1,Medallion architecture schema creation
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze_SCD")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver_SCD")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold_SCD")

In [0]:
 # load the source 

df = spark.read.format('csv').option('header','true').option('inferSchema','true').load("/Volumes/workspace/bronze_scd/sales_source/source_1.csv")
df.display()

In [0]:
# COMMAND ----------

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# --------------------------
# 1️⃣ BRONZE LAYER
# --------------------------

df_bronze = spark.read.csv(
    "/Volumes/workspace/bronze_scd/sales_source/source_1.csv",
    header=True,
    inferSchema=True
)

# Bronze table (append mode for incremental)
df_bronze.write.format("delta").mode("append").saveAsTable("bronze_SCD.bronze_sales_scd")

In [0]:
%sql
select * from bronze_SCD.bronze_sales_scd

In [0]:
# --------------------------
# 2️⃣ SILVER LAYER
# --------------------------

bronze_df = spark.table("bronze_SCD.bronze_sales_scd")

silver_new = bronze_df \
    .withColumn("quantity_measure",
        when(col("quantity") <= 1, "low")
        .when(col("quantity") <= 5, "medium")
        .otherwise("high")
    ) \
    .withColumn("price_measure",
        when(col("price") <= 170, "cheap")
        .when(col("price") <= 200, "medium")
        .otherwise("expensive")
    )
display(silver_new)

In [0]:
# Business hash for deduplication
business_cols = [
    "order_id", "customer_name", "product", "quantity", "price",
    "time", "quantity_measure", "price_measure"
]

In [0]:
silver_new = silver_new.withColumn(
    "business_hash",
    sha2(concat_ws("||", *business_cols), 256)
).select(
    "order_id", "customer_name", "product", "quantity", "price",
    "time", "quantity_measure", "price_measure", "business_hash"
)
display(silver_new)

In [0]:
#silver_tab is variable and silver_SCD.iris_clean
silver_tab = "silver_SCD.sales_clean"
if not spark.catalog.tableExists(silver_tab):
    silver_new.write.format("delta").mode("append").saveAsTable(silver_tab)
else:
    silver_existing = spark.table(silver_tab)
    silver_increment = silver_new.join(silver_existing, "business_hash", "left_anti")
    silver_increment.write.format("delta").mode("append").saveAsTable(silver_tab)


In [0]:
%sql
select * from silver_SCD.sales_clean

In [0]:
# --------------------------
# 3️⃣ GOLD LAYER (SCD Type-2)
# --------------------------

gold_table_name = "gold_SCD.sales_dim"
silver_df = spark.table(silver_tab)

# Create Gold table dynamically if first run
if not spark.catalog.tableExists(gold_table_name):
    silver_df.withColumn("valid_from", current_timestamp()) \
             .withColumn("valid_to", lit(None).cast("timestamp")) \
             .withColumn("valid_flag", lit("Y")) \
             .withColumn("updated", current_timestamp()) \
             .write.format("delta").mode("overwrite").saveAsTable(gold_table_name)

gold_tbl = DeltaTable.forName(spark, gold_table_name)

In [0]:
%sql
select * from gold_SCD.sales_dim

In [0]:
# Identify changed/new rows
changes_df = silver_df.alias("src") \
.join(gold_tbl.toDF().alias("tgt"),
      (col("src.order_id") == col("tgt.order_id")) &
      (col("tgt.valid_flag") == "Y"),
      "left") \
.where(col("tgt.business_hash").isNull() | (col("src.business_hash") != col("tgt.business_hash"))) \
.select("src.*")

In [0]:
# Deduplicate source to avoid multiple MERGE matches
window = Window.partitionBy("order_id").orderBy(col("business_hash").desc())
deduped_changes = changes_df.withColumn("rn", row_number().over(window)) \
                            .filter(col("rn") == 1) \
                            .drop("rn")

In [0]:
if deduped_changes.limit(1).count() > 0:
    # 1️⃣ Expire old rows (only update metadata)
    gold_tbl.alias("tgt").merge(
        deduped_changes.alias("src"),
        "tgt.order_id = src.order_id AND tgt.valid_flag = 'Y'"
    ).whenMatchedUpdate(set={
        "valid_to": current_timestamp(),
        "valid_flag": lit("N"),
        "updated": current_timestamp()
    }).execute()

    # 2️⃣ Insert new version rows
    new_df = deduped_changes \
        .withColumn("valid_from", current_timestamp()) \
        .withColumn("valid_to", lit(None).cast("timestamp")) \
        .withColumn("valid_flag", lit("Y")) \
        .withColumn("updated", current_timestamp())

    new_df.write.format("delta").mode("append").saveAsTable(gold_table_name)

In [0]:
%sql
select * from workspace.gold_scd.sales_dim